## Quiz 05 - Parallel Computing, Reproducibility, and Containers

### Instructions

This quiz is based on the material covered in lectures 20 to 24. You may use any resources available to you, including the lecture notes and the internet.

All the data required for this quiz can be found in the `data` folder within this repository. If you need to recreate the datasets, you can do so by running the Python script included in the `script-data-generation` folder.

**Important:** Please start by completing Question 01 to set up the correct Python environment before proceeding with the other questions.

Answer the questions in the cells below.
When you are done, please submit your work as an `.html` file on Canvas, or share a link to a GitHub repository with your answers.

### **Question 01: Setting up the Python Environment**

Before proceeding with the rest of the quiz, it is important to set up a Python environment with specific package versions to ensure compatibility and reproducibility. This quiz requires **Python 3.12** and the following packages with exact versions:
- `dask=2026.3.0`
- `duckdb=1.5.1`
- `ipykernel=7.2.0`
- `joblib=1.5.3`
- `numpy=2.4.3`
- `pandas=3.0.2`
- `pyarrow=23.0.1`

You can use tools like `conda`, `pipenv`, or `uv` to manage your environment. If you use conda (recommended), please make sure you **create the environment and install all packages in the same command**. Also include `-c conda-forge` in your command. Make sure to change your current environment to the new environment after creation.

Write the terminal commands in the code cell below:

In [17]:
# Please write your bash commands here. You can run them using the `!` operator or the `%%bash` magic.
import sys
import platform

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())

Python executable: /opt/homebrew/Cellar/micromamba/2.5.0_4/envs/quiz05-env/bin/python
Python version: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 17:06:14) [Clang 19.1.7 ]
Platform: macOS-14.1.1-arm64-arm-64bit


### Environment Setup Issues and Resolution

While setting up the Python environment using conda inside Jupyter, we encountered significant delays during the dependency resolution step ("Solving environment"), which caused the process to hang or retry multiple times. Additionally, there were issues with activating the environment within Jupyter (`conda activate` not working inside `%%bash`) and the kernel not appearing correctly in VS Code. 

To resolve this, we switched to using **micromamba**, a faster alternative to conda with a more efficient dependency solver. We created the environment directly in the terminal, initialized the shell for micromamba, activated the environment, and then manually registered the Jupyter kernel. After restarting VS Code, the correct Python 3.12 kernel was successfully detected and used for the notebook.

### Question 02: Understanding the `map` Function and Parallelism

The built-in Python `map()` function applies a function to each element sequentially. Using `joblib`, rewrite the following serial code to run in parallel using **all available cores**. Compare the results to verify correctness.

```python
import numpy as np

def cube_root(x):
    return x ** (1/3)

numbers = np.arange(1, 500001)

# Serial version using map
serial_result = list(map(cube_root, numbers))
print("First 5 serial results:", serial_result[:5])
```

In [4]:
import numpy as np

def cube_root(x):
    return x ** (1/3)

numbers = np.arange(1, 500001)

# Serial version using map
serial_result = list(map(cube_root, numbers))
print("First 5 serial results:", serial_result[:5])

First 5 serial results: [np.float64(1.0), np.float64(1.2599210498948732), np.float64(1.4422495703074083), np.float64(1.5874010519681994), np.float64(1.7099759466766968)]


In [5]:
# Please write your answer here.
from joblib import Parallel, delayed

# Parallel version using joblib
parallel_result = Parallel(n_jobs=-1)(
    delayed(cube_root)(x) for x in numbers
)

print("First 5 parallel results:", parallel_result[:5])

# Verify correctness
print("Results match:", np.allclose(serial_result, parallel_result))

python(3939) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(3941) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(3942) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(3943) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(3944) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(3945) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(3946) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(3947) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(3948) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(3949) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


First 5 parallel results: [np.float64(1.0), np.float64(1.2599210498948732), np.float64(1.4422495703074083), np.float64(1.5874010519681994), np.float64(1.7099759466766968)]
Results match: True


### Question 03: Measuring Parallel Speedup

Create a function called `simulate_computation` that generates 100,000 random numbers and calculates their variance. Using `%timeit`, measure and compare the execution time of:

1. Running the function **4 times sequentially** in a list comprehension (`[simulate_computation() for _ in range(4)]`)
2. Running the function **4 times in parallel** using `joblib` with 4 workers

Print and compare both timing results.

In [8]:
import numpy as np
from joblib import Parallel, delayed

# Function to simulate computation
def simulate_computation():
    data = np.random.rand(100000)
    return np.var(data)

# Sequential timing
%timeit [simulate_computation() for _ in range(4)]

# Parallel timing (4 workers)
%timeit Parallel(n_jobs=4)([delayed(simulate_computation)() for _ in range(4)])

3.15 ms ± 336 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


python(4142) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(4143) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(4144) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(4145) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


28.3 ms ± 14.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


### Question 04: Dask Array with Custom Chunk Sizes

Create a Dask array of shape (5000, 2000) filled with random integers between 1 and 100. Use chunks of size (500, 500). Then:

1. Compute the sum of each row
2. Calculate the mean and standard deviation of the entire array
3. Print all three results

In [9]:
# Please write your answer here.
import dask.array as da

# Create Dask array
arr = da.random.randint(1, 101, size=(5000, 2000), chunks=(500, 500))

# 1. Sum of each row
row_sums = arr.sum(axis=1)

# 2. Mean and standard deviation of entire array
mean_val = arr.mean()
std_val = arr.std()

# Compute results
row_sums_result = row_sums.compute()
mean_result = mean_val.compute()
std_result = std_val.compute()

# Print results
print("First 5 row sums:", row_sums_result[:5])
print("Mean:", mean_result)
print("Standard Deviation:", std_result)

First 5 row sums: [ 99704 100496 100708 100154 103208]
Mean: 50.5008446
Standard Deviation: 28.86829656711062


### Question 05: Optimising Chunk Size

The chunk size significantly affects Dask performance. Create a Dask array with 100,000 random numbers and test three different chunk sizes: 1,000 (many small chunks), 10,000 (medium chunks), and 50,000 (few large chunks).

For each configuration, measure the time to compute `mean(sin(x) + cos(x))`. Which chunk size performed best? Explain why in a comment.

In [10]:
# Please write your answer here.
import dask.array as da
import numpy as np

# Function to test chunk size performance
def test_chunk(chunk_size):
    x = da.random.random(100000, chunks=chunk_size)
    return (da.sin(x) + da.cos(x)).mean().compute()

# Chunk sizes
chunk_sizes = [1000, 10000, 50000]

# Measure time
for cs in chunk_sizes:
    print(f"\nChunk size: {cs}")
    %timeit test_chunk(cs)


Chunk size: 1000
81.5 ms ± 11.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)

Chunk size: 10000
11.1 ms ± 1.55 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)

Chunk size: 50000
5.31 ms ± 449 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [ ]:
# Smaller chunk sizes (e.g., 1000) introduce higher overhead due to more task scheduling,
# while very large chunks (e.g., 50000) reduce parallelism. Medium chunk sizes (e.g., 10000)
# typically perform best because they balance parallelism and overhead efficiently.

### Question 06: Reading Parquet Files with Column Selection

The `data` folder contains Parquet files for multiple countries. Using Dask, read **all Parquet files at once** (`data/*.parquet`), but load only the `year` and `population` columns.

Calculate the total world population for each year across all countries and display the results sorted by year.

In [11]:
# Please write your answer here.
import dask.dataframe as dd

# Read all parquet files, selecting only required columns
df = dd.read_parquet("data/*.parquet", columns=["year", "population"])

# Group by year and sum population
result = df.groupby("year")["population"].sum().compute()

# Sort results
result = result.sort_index()

print(result)


year
1945     566994202
1946     596804909
1947     606569895
1948     637303888
1949     644613118
           ...    
2019    1893887207
2020    1959057915
2021    1928046753
2022    1985837056
2023    1980706538
Name: population, Length: 79, dtype: int64


### Question 07: DuckDB with Multiple Conditions

Load the `data.csv` file into a Pandas DataFrame. Using DuckDB, write a SQL query that:

1. Selects countries where `gdp_per_capita` was between 10000 and 50000
2. Filters for years between 2000 and 2020
3. Orders results by `gdp_per_capita` in descending order
4. Limits to the top 2 results

Execute the query and display the results.

In [12]:
# Please write your answer here.
import pandas as pd
import duckdb

# Load CSV into pandas
df = pd.read_csv("data/data.csv")

# Run DuckDB query
result = duckdb.query("""
SELECT *
FROM df
WHERE gdp_per_capita BETWEEN 10000 AND 50000
  AND year BETWEEN 2000 AND 2020
ORDER BY gdp_per_capita DESC
LIMIT 2
""").to_df()

print(result)

  country  year  gdp_per_capita  population
0     USA  2002    48942.492140   284207485
1     USA  2003    47607.365171   277711486


### Question 08: DuckDB with Aggregation

Using the same `data.csv` file, write a SQL query with DuckDB that calculates:

1. The average GDP per capita for each country
2. The minimum and maximum years in the dataset for each country

Group by country and display all results.

In [13]:
# Please write your answer here.
# DuckDB aggregation query
result = duckdb.query("""
SELECT 
    country,
    AVG(gdp_per_capita) AS avg_gdp_per_capita,
    MIN(year) AS min_year,
    MAX(year) AS max_year
FROM df
GROUP BY country
""").to_df()

print(result)

  country  avg_gdp_per_capita  min_year  max_year
0      UK        27496.851363      1945      2023
1  Brazil         5496.292031      1945      2023
2   India         1251.704443      1945      2023
3     USA        40189.822290      1945      2023


### Question 09: Generating `requirements.txt` and `environment.yml` Files

Write the commands to:

1. Export your current environment's packages to a `requirements.txt` and an `environment.yml` file
2. Show how someone else would install these exact dependencies in these two cases

Explain each step with comments. It is not necessary to run the code.

In [14]:
# Please write your answer here.
# Export current environment to requirements.txt (pip-style)
# This lists all installed packages with exact versions
# Useful for recreating environment using pip
!pip freeze > requirements.txt

# Export current conda/mamba environment to environment.yml
# This includes Python version + channels + dependencies
# Useful for recreating environment using conda/mamba
!conda env export --name quiz05-env > environment.yml


# ----- How someone else would recreate the environment -----

# Using requirements.txt (pip)
# 1. Create a new environment
# 2. Install dependencies from file
# python -m venv new_env
# source new_env/bin/activate
# pip install -r requirements.txt

# Using environment.yml (conda/mamba)
# 1. Create environment directly from file
# conda env create -f environment.yml
# 2. Activate it
# conda activate quiz05-env

python(4468) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(4472) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


### Question 10: Troubleshooting a Broken Dockerfile

The following Dockerfile has several errors. Identify and fix 5 issues, then explain what was wrong with each line:

```dockerfile
# Broken Dockerfile - Fix the errors
from ubuntu

RUN apt install python3 python3-pip
RUN pip install numpy pandas

COPY . .
EXPOSE 8888
RUN ["python3", "app.py"]
```

Write the corrected Dockerfile and list each error with its fix.

### Corrected Dockerfile

```dockerfile
FROM ubuntu:22.04

ENV DEBIAN_FRONTEND=noninteractive

RUN apt-get update && apt-get install -y python3 python3-pip

RUN pip3 install numpy pandas

WORKDIR /app

COPY . .

EXPOSE 8888

CMD ["python3", "app.py"]

### Dockerfile Errors and Fixes

1. `from ubuntu` → should be `FROM ubuntu:22.04`  
   Fix: Docker instructions must be uppercase and include a version for reproducibility.

2. `RUN apt install python3 python3-pip`  
   Fix: Missing `apt-get update` and `-y`. Correct: `apt-get update && apt-get install -y ...`

3. `RUN pip install numpy pandas`  
   Fix: Should use `pip3` to match Python3.

4. `COPY . .` without WORKDIR  
   Fix: Add `WORKDIR /app` before copying files.

5. `RUN ["python3", "app.py"]`  
   Fix: `RUN` is for build time. Should use `CMD` to run at container start.

6. Missing environment config  
   Fix: Add `ENV DEBIAN_FRONTEND=noninteractive` to avoid install prompts.

### Question 11 - Writing a Dockerfile to Install Software on a Base Image

Create a Dockerfile that starts from an Ubuntu image and installs the following software:

- Git version 1:2.43.0-1ubuntu7.3
- SQLite version 3.45.1-1ubuntu2

Ensure that you specify the exact versions of the packages by checking their versions after installation. Include commands to clean up the package manager cache after installation to reduce the image size.

### Q11: Dockerfile with Specific Package Versions

```dockerfile
FROM ubuntu:22.04

ENV DEBIAN_FRONTEND=noninteractive

# Update and install specific versions
RUN apt-get update && \
    apt-get install -y \
    git=1:2.34.1-1ubuntu1.10 \
    sqlite3=3.37.2-2ubuntu0.3 && \
    apt-get clean && \
    rm -rf /var/lib/apt/lists/*

# Verify versions
RUN git --version && sqlite3 --version

#### Please write your anwer here. You can use ```dockerfile to format your code

### Question 12: Dockerfile for a Jupyter Data Science Environment

Create a Dockerfile starting from Ubuntu that:

1. Installs Python 3.12 and pip
2. Installs `jupyterlab`, `numpy`, `pandas`, `matplotlib`, and `scikit-learn` with specific versions of your choice
3. Sets the working directory to `/home/analyst/notebooks`
4. Exposes port 8888
5. Starts JupyterLab with `--no-browser` and `--ip=0.0.0.0`

Clean up apt cache to reduce image size.

### Q12: Jupyter Data Science Dockerfile

```dockerfile
FROM ubuntu:22.04

ENV DEBIAN_FRONTEND=noninteractive

# Install Python 3.12 and pip
RUN apt-get update && \
    apt-get install -y python3.12 python3-pip && \
    apt-get clean && \
    rm -rf /var/lib/apt/lists/*

# Install required Python packages
RUN pip3 install \
    jupyterlab==4.2.0 \
    numpy==2.4.3 \
    pandas==3.0.2 \
    matplotlib==3.9.0 \
    scikit-learn==1.5.0

# Set working directory
WORKDIR /home/analyst/notebooks

# Expose port
EXPOSE 8888

# Start JupyterLab
CMD ["jupyter-lab", "--no-browser", "--ip=0.0.0.0"]

#### Please write your answer here. You can use ```dockerfile to format your code